In [ ]:
# 필요 시 설치
%pip install pandas scipy numpy

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import time

In [24]:
CSV_PATH = r"C:\Users\USER\OneDrive\바탕 화면\Ckrasseck\Study\주전공\3-1\동양중세사\소논문\데이터\kangxi_master_1661_1722.csv"   # ← 파일 경로 수정
CHUNK_SIZE = 50_000             # ← 메모리 상황에 따라 조절 (행 단위)


chunks = []
start = time.time()

for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)):
    # ✅ Null → 0 처리
    chunk = chunk.fillna(0)
    chunks.append(chunk)
    print(f"  청크 {i+1} 로드 완료 | 행: {len(chunk)}")

df = pd.concat(chunks, ignore_index=True)

print(f"\n✅ 전체 로딩 완료")
print(f"   Shape : {df.shape}")
print(f"   소요시간 : {time.time() - start:.2f}초")
print(f"   메모리 사용량 : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


  청크 1 로드 완료 | 행: 35171

✅ 전체 로딩 완료
   Shape : (35171, 760)
   소요시간 : 2.13초
   메모리 사용량 : 206.0 MB


In [26]:
# 마지막행 확인
print(df.tail(-210))

       한자단어  1661_01  1661_02  1661_03  1661_04  1661_05  1661_06  1661_07  \
210     祭太歲      1.0      0.0      0.0      0.0      0.0      0.0      0.0   
211      之神      1.0      0.0      0.0      0.0      0.0      0.0      0.0   
212      위기      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
213       錢      0.0      9.0      0.0      0.0      0.0      1.0      0.0   
214      拖欠      0.0      8.0      0.0      0.0      0.0      0.0      0.0   
...     ...      ...      ...      ...      ...      ...      ...      ...   
35166  統寶坻縣      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
35167  伍爾泰為      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
35168  統納爾賽      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
35169  統正紅旗      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
35170    那為      0.0      0.0      0.0      0.0      0.0      0.0      0.0   

       1661_08  1661_09  ...  1722_02  1722_03  1722_04  1722_0

In [27]:
# ── Case A ──────────────────────────────────────────
# object 컬럼을 인덱스로 빼고, 나머지 숫자만 희소행렬로

# object 컬럼과 숫자 컬럼 분리
obj_cols  = df.select_dtypes(include='object').columns.tolist()
num_cols  = df.select_dtypes(exclude='object').columns.tolist()

# 라벨은 따로 저장
label_df  = df[obj_cols].copy()
num_df    = df[num_cols].copy()

print(f"라벨 컬럼 : {obj_cols}")
print(f"숫자 컬럼 수 : {len(num_cols)}")
print(label_df.head(3))

# ✅ 숫자 컬럼만 희소행렬 변환
sparse_matrix = csr_matrix(num_df.values.astype(np.float32))

print(f"\n✅ 희소행렬 변환 완료 | Shape: {sparse_matrix.shape}")
print(f"   Non-zero 원소: {sparse_matrix.nnz:,}")


C:\Users\USER\AppData\Local\Temp\ipykernel_18420\3794085145.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols  = df.select_dtypes(include='object').columns.tolist()


라벨 컬럼 : ['한자단어']
숫자 컬럼 수 : 759
  한자단어
0    等
1   皇帝
2   大臣

✅ 희소행렬 변환 완료 | Shape: (35171, 759)
   Non-zero 원소: 110,573


In [40]:
print(label_df.tail(-210))

       한자단어
210     祭太歲
211      之神
212      위기
213       錢
214      拖欠
...     ...
35166  統寶坻縣
35167  伍爾泰為
35168  統納爾賽
35169  統正紅旗
35170    那為

[34961 rows x 1 columns]


In [28]:
# DataFrame → numpy → CSR 희소행렬
sparse_matrix = csr_matrix(num_df.values.astype(np.float32))

print(f"✅ 희소행렬 변환 완료")
print(f"   Shape  : {sparse_matrix.shape}")
print(f"   저장된 원소 수 (non-zero) : {sparse_matrix.nnz:,}")
print(f"   희소율 (sparsity) : {1 - sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]):.2%}")


✅ 희소행렬 변환 완료
   Shape  : (35171, 759)
   저장된 원소 수 (non-zero) : 110,573
   희소율 (sparsity) : 99.59%


In [29]:
from scipy.sparse import save_npz, load_npz

# 저장 (다음번엔 CSV 다시 안 읽어도 됨 — 훨씬 빠름)
save_npz("sparse_matrix.npz", sparse_matrix)
print("✅ .npz 저장 완료")

# 다시 불러올 때
# sparse_matrix = load_npz("sparse_matrix.npz")


✅ .npz 저장 완료


In [44]:
# '위기'라는 값이 들어있는 컬럼 찾기
for col in label_df.columns:
    if label_df[col].astype(str).str.contains('위기').any():
        print(f"✅ 발견! 컬럼명: '{col}'")
        print(f"   고유값: {label_df[col].unique()[:10]}")


✅ 발견! 컬럼명: '한자단어'
   고유값: <StringArray>
['等', '皇帝', '大臣', '索尼', '諸王', '曰', '先', '不', '侍衛', '皇太后']
Length: 10, dtype: str


In [45]:
# 인덱스 세팅
num_df.index = label_df['한자단어'].values

# ① '위기' 행 → y (타겟)
y_row = num_df.loc['위기']                     # shape: (n_samples,)

# ② 나머지 35,169개 행 → X (피처)
X_df = num_df.drop(index='위기')               # shape: (35169, n_samples)

# ③ 전치: 행/열 뒤집기
X = X_df.T                                     # shape: (n_samples, 35169)
y_raw = y_row.values                           # shape: (n_samples,)

print(f"X shape : {X.shape}   → (샘플 수, 단어 수)")
print(f"y shape : {y_raw.shape}")
print(f"y 값 샘플 : {y_raw[:10]}")


X shape : (759, 35170)   → (샘플 수, 단어 수)
y shape : (759,)
y 값 샘플 : [0. 0. 0. 0. 0. 0. 0. 0. 1. 1.]


In [46]:
# y값이 연속형(빈도/횟수)이면 0/1로 변환
y = (y_raw > 0).astype(int)

print(f"y=0 (위기 없음) : {(y==0).sum():,}개")
print(f"y=1 (위기 있음) : {(y==1).sum():,}개")


y=0 (위기 없음) : 707개
y=1 (위기 있음) : 52개


In [ ]:
%pip install scikit-learn

In [52]:
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ── 1. X, y 세팅 ──────────────────────────────────
num_df.index = label_df['한자단어'].values

y = num_df.loc['위기'].values.astype(int)          # 이진 (0/1)
X_df = num_df.drop(index='위기').T                 # (샘플 수, 35169)
X = csr_matrix(X_df.values.astype(np.float32))

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"y=0 : {(y==0).sum():,}개  |  y=1 : {(y==1).sum():,}개")

# ── 2. Train / Test 분할 ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y              # 클래스 비율 유지
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# ── 3. 로지스틱 학습 ──────────────────────────────
model = LogisticRegression(
    solver='saga',
    max_iter=1000,
    class_weight='balanced',   # ✅ 0/1 불균형 자동 보정
    n_jobs=-1,
    verbose=1
)
model.fit(X_train, y_train)

# ── 4. 평가 ───────────────────────────────────────
y_pred = model.predict(X_test)

print(f"\n정확도 : {accuracy_score(y_test, y_pred):.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['위기없음(0)', '위기있음(1)']))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))


X shape : (759, 35170)
y shape : (759,)
y=0 : 707개  |  y=1 : 52개

Train: (607, 35170)  |  Test: (152, 35170)


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


max_iter reached after 1 seconds

정확도 : 0.9276

=== Classification Report ===
              precision    recall  f1-score   support

     위기없음(0)       0.93      0.99      0.96       142
     위기있음(1)       0.00      0.00      0.00        10

    accuracy                           0.93       152
   macro avg       0.47      0.50      0.48       152
weighted avg       0.87      0.93      0.90       152

=== Confusion Matrix ===
[[141   1]
 [ 10   0]]


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model_lasso = LogisticRegression(
    penalty='l1',              # ✅ Lasso (L1 규제)
    solver='saga',             # L1 + 희소행렬 지원하는 유일한 solver
    C=1.0,                     # 규제 강도 (작을수록 강한 규제)
    class_weight='balanced',   # 클래스 불균형 보정
    max_iter=1000,
    n_jobs=-1,
    verbose=1
)

model_lasso.fit(X_train, y_train)
y_pred = model_lasso.predict(X_test)

print(f"\n=== Classification Report ===")
print(f"\n정확도 : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred,
      target_names=['위기없음(0)', '위기있음(1)']))
print(f"=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


max_iter reached after 4 seconds

=== Classification Report ===

정확도 : 0.9013
              precision    recall  f1-score   support

     위기없음(0)       0.94      0.96      0.95       142
     위기있음(1)       0.14      0.10      0.12        10

    accuracy                           0.90       152
   macro avg       0.54      0.53      0.53       152
weighted avg       0.89      0.90      0.89       152

=== Confusion Matrix ===
[[136   6]
 [  9   1]]


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [54]:
import pandas as pd
import numpy as np

# 계수가 0이 아닌 단어 = Lasso가 선택한 단어
coef = model_lasso.coef_[0]
word_names = X_df.columns.tolist()   # 한자단어 35169개

importance_df = pd.DataFrame({
    '한자단어': word_names,
    '계수': coef
})

# 0이 아닌 것만 필터링
selected = importance_df[importance_df['계수'] != 0]
selected = selected.reindex(selected['계수'].abs().sort_values(ascending=False).index)

print(f"전체 단어 수     : {len(word_names):,}개")
print(f"Lasso 선택 단어  : {len(selected):,}개")
print(f"\n=== 위기 예측에 중요한 상위 20개 한자 ===")
print(selected.head(20).to_string(index=False))


전체 단어 수     : 35,170개
Lasso 선택 단어  : 826개

=== 위기 예측에 중요한 상위 20개 한자 ===
한자단어        계수
 翰林院  0.535703
   駐  0.523914
  乙未  0.483587
  刑部  0.478519
  詳閱  0.478115
   內  0.468932
  江南 -0.468053
  二月 -0.465065
  江西  0.464621
  災額 -0.448701
  戊寅 -0.444731
 尼哈番  0.439060
   賊  0.438392
 十四年  0.424402
  皇上  0.421825
  成都  0.420884
  丹巴  0.419060
   每  0.406147
   俟 -0.402769
   疏 -0.402369


In [56]:
import numpy as np
from sklearn.metrics import f1_score

# 확률값으로 예측
y_prob = model_lasso.predict_proba(X_test)[:, 1]

# 임계값별 F1 스캔
thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    results.append({'threshold': round(t, 2), 'f1': round(f1, 4)})

result_df = pd.DataFrame(results)
print(result_df)

# 최적 임계값 적용
best_t = result_df.loc[result_df['f1'].idxmax(), 'threshold']
print(f"\n✅ 최적 임계값 : {best_t}")
y_pred_best = (y_prob >= best_t).astype(int)
print(classification_report(y_test, y_pred_best,
      target_names=['위기없음(0)', '위기있음(1)']))


    threshold      f1
0        0.10  0.0988
1        0.15  0.1231
2        0.20  0.1277
3        0.25  0.0909
4        0.30  0.1212
5        0.35  0.0741
6        0.40  0.0952
7        0.45  0.1111
8        0.50  0.1176
9        0.55  0.1250
10       0.60  0.1250
11       0.65  0.1429
12       0.70  0.1538
13       0.75  0.0000
14       0.80  0.0000
15       0.85  0.0000

✅ 최적 임계값 : 0.7
              precision    recall  f1-score   support

     위기없음(0)       0.94      0.99      0.96       142
     위기있음(1)       0.33      0.10      0.15        10

    accuracy                           0.93       152
   macro avg       0.64      0.54      0.56       152
weighted avg       0.90      0.93      0.91       152



In [ ]:
%pip install imblearn

In [59]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"SMOTE 전 → y=0: {(y_train==0).sum():,}  y=1: {(y_train==1).sum():,}")
print(f"SMOTE 후 → y=0: {(y_train_sm==0).sum():,}  y=1: {(y_train_sm==1).sum():,}")

model_sm = LogisticRegression(
    penalty='l1', solver='saga', C=1.0,
    max_iter=1000, n_jobs=-1, verbose=1
)
model_sm.fit(X_train_sm, y_train_sm)

print(classification_report(y_test, model_sm.predict(X_test),
      target_names=['위기없음(0)', '위기있음(1)']))


SMOTE 전 → y=0: 565  y=1: 42
SMOTE 후 → y=0: 565  y=1: 565


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


max_iter reached after 8 seconds
              precision    recall  f1-score   support

     위기없음(0)       0.94      0.96      0.95       142
     위기있음(1)       0.17      0.10      0.12        10

    accuracy                           0.91       152
   macro avg       0.55      0.53      0.54       152
weighted avg       0.89      0.91      0.90       152



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

for c in [0.01, 0.1, 1.0, 10.0]:
    m = LogisticRegression(
        penalty='l1', solver='saga', C=c,
        class_weight='balanced', max_iter=500, n_jobs=-1
    )
    score = cross_val_score(m, X_train, y_train,
                            cv=StratifiedKFold(5),
                            scoring='f1').mean()
    print(f"C={c:5}  →  F1: {score:.4f}")


In [61]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model_rf = RandomForestClassifier(
    n_estimators=300,          # 트리 수
    class_weight='balanced',   # ✅ 불균형 자동 보정
    random_state=42,
    n_jobs=-1,
    verbose=1
)

model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf,
      target_names=['위기없음(0)', '위기있음(1)']))


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.3s


              precision    recall  f1-score   support

     위기없음(0)       0.93      1.00      0.97       142
     위기있음(1)       0.00      0.00      0.00        10

    accuracy                           0.93       152
   macro avg       0.47      0.50      0.48       152
weighted avg       0.87      0.93      0.90       152



[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:    0.6s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:    0.0s finished
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [73]:
import pandas as pd
import numpy as np

# 피처 중요도 추출
word_names = X_df.columns.tolist()
importances = model_rf.feature_importances_

importance_df = pd.DataFrame({
    '한자단어': word_names,
    '중요도': importances
}).sort_values('중요도', ascending=False)

print("=== 위기 예측 상위 30개 한자 ===")
print(importance_df.head(5).to_string(index=False))

# 중요도 상위 N개만 선택
TOP_N = 1500    # ← 조절 가능
top_words = importance_df.head(TOP_N)['한자단어'].tolist()

from scipy.sparse import csr_matrix

# 상위 N개 컬럼만 추출
X_top = csr_matrix(X_df[top_words].values.astype(np.float32))

X_tr, X_te, y_tr, y_te = train_test_split(
    X_top, y, test_size=0.2,
    random_state=42, stratify=y
)

# 랜덤포레스트 재학습
model_rf2 = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model_rf2.fit(X_tr, y_tr)

print(classification_report(y_te, model_rf2.predict(X_te),
      target_names=['위기없음(0)', '위기있음(1)']))




=== 위기 예측 상위 30개 한자 ===
한자단어      중요도
   偽 0.004368
   於 0.003769
   上 0.003594
 十四年 0.003545
  乙未 0.003250
              precision    recall  f1-score   support

     위기없음(0)       0.93      1.00      0.97       142
     위기있음(1)       0.00      0.00      0.00        10

    accuracy                           0.93       152
   macro avg       0.47      0.50      0.48       152
weighted avg       0.87      0.93      0.90       152



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [74]:
%pip install lightgbm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [75]:
import lightgbm as lgb
from sklearn.metrics import classification_report

# 불균형 비율 계산
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"불균형 비율 (neg/pos) : {scale:.1f}")

model_lgb = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale,    # ✅ 불균형 자동 보정
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=True)]  # 과적합 방지
)

y_pred_lgb = model_lgb.predict(X_test)
print(classification_report(y_test, y_pred_lgb,
      target_names=['위기없음(0)', '위기있음(1)']))


불균형 비율 (neg/pos) : 13.5
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.23892
              precision    recall  f1-score   support

     위기없음(0)       0.93      1.00      0.97       142
     위기있음(1)       0.00      0.00      0.00        10

    accuracy                           0.93       152
   macro avg       0.47      0.50      0.48       152
weighted avg       0.87      0.93      0.90       152



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\

In [77]:
import pandas as pd
import numpy as np

importance_df = pd.DataFrame({
    '한자단어': X_df.columns.tolist(),
    '중요도': model_lgb.feature_importances_
}).sort_values('중요도', ascending=False)

print("=== 위기 예측 상위 30개 한자 ===")
print(importance_df.head(5).to_string(index=False))

from sklearn.metrics import f1_score

y_prob_lgb = model_lgb.predict_proba(X_test)[:, 1]

best_f1, best_t = 0, 0.5
for t in np.arange(0.1, 0.9, 0.05):
    f1 = f1_score(y_test, (y_prob_lgb >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"✅ 최적 임계값 : {best_t:.2f}  →  F1 : {best_f1:.4f}")
y_pred_final = (y_prob_lgb >= best_t).astype(int)
print(classification_report(y_test, y_pred_final,
      target_names=['위기없음(0)', '위기있음(1)']))


=== 위기 예측 상위 30개 한자 ===
한자단어  중요도
   因    1
   駐    1
   等    1
  康熙    1
   亦    1
✅ 최적 임계값 : 0.10  →  F1 : 0.1739
              precision    recall  f1-score   support

     위기없음(0)       0.95      0.77      0.85       142
     위기있음(1)       0.11      0.40      0.17        10

    accuracy                           0.75       152
   macro avg       0.53      0.59      0.51       152
weighted avg       0.89      0.75      0.81       152



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [89]:
from sklearn.decomposition import TruncatedSVD
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.metrics import classification_report
import numpy as np

# ── 1. 차원 축소 (35,169 → 155) ──────────────────
svd = TruncatedSVD(n_components=155, random_state=42)  # ← 조절 가능
X_svd = svd.fit_transform(X)   # (750, 155)

print(f"차원 축소 완료 : {X.shape} → {X_svd.shape}")
print(f"설명된 분산 비율 : {svd.explained_variance_ratio_.sum():.2%}")

# ── 2. Train/Test 분할 ────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_svd, y, test_size=0.2,
    random_state=42, stratify=y
)

# ── 3. SMOTE ──────────────────────────────────────
smote = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
print(f"\nSMOTE 후 → y=0: {(y_tr_sm==0).sum()}  y=1: {(y_tr_sm==1).sum()}")

# ── 4. LightGBM 재학습 ────────────────────────────
model_final = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model_final.fit(X_tr_sm, y_tr_sm)

# ── 5. 임계값 최적화 ──────────────────────────────
from sklearn.metrics import f1_score
y_prob = model_final.predict_proba(X_te)[:, 1]

best_f1, best_t = 0, 0.5
for t in np.arange(0.1, 0.9, 0.05):
    f1 = f1_score(y_te, (y_prob >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"\n✅ 최적 임계값 : {best_t:.2f}  →  F1 : {best_f1:.4f}")
print(classification_report(y_te, (y_prob >= best_t).astype(int),
      target_names=['위기없음(0)', '위기있음(1)']))


차원 축소 완료 : (759, 35170) → (759, 155)
설명된 분산 비율 : 74.50%

SMOTE 후 → y=0: 565  y=1: 565

✅ 최적 임계값 : 0.50  →  F1 : 0.1538
              precision    recall  f1-score   support

     위기없음(0)       0.94      0.99      0.96       142
     위기있음(1)       0.33      0.10      0.15        10

    accuracy                           0.93       152
   macro avg       0.64      0.54      0.56       152
weighted avg       0.90      0.93      0.91       152



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [79]:
# 최적 컴포넌트 수 탐색
for n in [50, 100, 200, 300]:
    svd_t = TruncatedSVD(n_components=n, random_state=42)
    X_t = svd_t.fit_transform(X)
    print(f"n={n:3d}  설명 분산: {svd_t.explained_variance_ratio_.sum():.2%}")


n= 50  설명 분산: 54.48%
n=100  설명 분산: 66.18%
n=200  설명 분산: 79.60%
n=300  설명 분산: 87.72%
